In [8]:
import os
from pathlib import Path

from f_dirs import get_data_dirs
dirs = get_data_dirs()

cwd = os.getcwd()
print(f"Current working directory: {cwd}")

# Print every item in the dirs dictionary
# But 'DirPaths' object has no attribute 'items'
# Iterate over them
for attr in dir(dirs):
    if attr.startswith('_'):
        continue
    if callable(getattr(dirs, attr)):
        continue

    print(f"{attr}: {getattr(dirs, attr)}")

Root directory: c:\Users\lazyst\Files\ucl\Dissertation\build
Current working directory: c:\Users\lazyst\Files\ucl\Dissertation\build\src
data_dir: D:\FAME - LN dataset\Dropbox\fame_clean\1_FAME_raw_data\2025.07.30
output_dir: c:\Users\lazyst\Files\ucl\Dissertation\build\build\output
raw_data_dir: D:\FAME - LN dataset\Dropbox\fame_clean\1_FAME_raw_data\2025.02
root_data_dir: D:\FAME - LN dataset\Dropbox\fame_clean
root_dir: c:\Users\lazyst\Files\ucl\Dissertation\build
work_dir: c:\Users\lazyst\Files\ucl\Dissertation\build\build\src


### 1. Generate the raw file dictionary

This script resolves the project data paths, scans the raw-data folder, builds a nested dictionary of file metadata by company and file category, and writes it to JSON.  
`get_data_dirs()` defines the working directories  
`build_raw_file_dict()` performs the recursive traversal and file collection through its helper functions.  

In [ ]:
from flask import json
import pandas as pd

from f_traverse import build_raw_file_dict, RawFileDict

pd.options.mode.chained_assignment = None  # default='warn'

if dirs.raw_data_dir is None:
	raise ValueError("raw_data_dir is None. Please check your .env file and ensure RAW_DATA_DIR is set correctly.")

raw_file_dict = build_raw_file_dict(dirs.raw_data_dir)
with open(dirs.output_dir / "raw_file_dict.json", "w") as f:
    json.dump(raw_file_dict, f, indent=4)
    print(f"✅ Successfully built raw file dictionary and saved to: {dirs.output_dir / 'raw_file_dict.json'}")

Traversing industry directory: company_year_consolidation_empl_flag.R
Traversing industry directory: 88
Traversing industry directory: 52
Traversing industry directory: 61
Traversing industry directory: 94
Traversing industry directory: 47
Traversing industry directory: 46
Traversing industry directory: 56
Traversing industry directory: 90
Traversing industry directory: 32
Traversing industry directory: 23
Traversing industry directory: 43
Traversing industry directory: 62
Traversing industry directory: 82
Traversing industry directory: 68
Traversing industry directory: 70
Traversing industry directory: 86
Traversing industry directory: 85
Traversing industry directory: 45
Traversing industry directory: 77
Traversing industry directory: 96
Traversing industry directory: 49
Traversing industry directory: 93
Traversing industry directory: 74
Traversing industry directory: 41
Traversing industry directory: 81
Traversing industry directory: 55
Traversing industry directory: 73
Traversing i

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\lazym\\Documents\\Code\\dissertation\\build\\src\\build\\output\\raw_file_dict.json'

### 2. Ensure database connection

In [ ]:
import pandas as pd
# Import ibis-framework
import ibis
# pip install 'ibis-framework[duckdb,examples]' # examples is only required to access the sample data Ibis provides

print("Path:", ibis.__file__)
print("Version:", getattr(ibis, "__version__", "No version found"))
pd.options.mode.chained_assignment = None  # default='warn'

db_path = dirs.output_dir / "fame_data.duckdb"

# 3. Define the explicit schema using Ibis types
# This provides type safety and allows your IDE to lint your definitions
fame_schema = ibis.schema({
    "company_name": "string",
    "registered_number": "string", 
    "ticker_symbol": "string",
    
    # Registered Office (R/O) Details
    "ro_address": "string",
    "ro_address_line_1": "string",
    "ro_address_line_2": "string",
    "ro_address_line_3": "string",
    "ro_address_line_4": "string",
    "ro_address_line_5": "string",
    "ro_city": "string",
    "ro_county": "string",
    "ro_postcode": "string",
    "ro_full_postcode": "string",
    "ro_country": "string",
    
    # Ingested as string due to raw DMS format (e.g., 54° 36' 24.3'' N)
    "ro_latitude": "string", 
    "ro_longitude": "string",
    
    "ro_nuts_region": "string",
    "ro_postal_region": "string",
    "ro_phone": "string",
    "ro_phone_registered_on_tps": "string",
    "ro_phone_registered_on_ctps": "string",
    
    # Primary Trading Address Details
    "primary_trading_address": "string",
    "primary_trading_address_latitude": "string",
    "primary_trading_address_longitude": "string",
    "primary_trading_address_no_of_employees": "string",
    
    # Operations & Classification
    "branch_name": "string",
    "trade_description": "string",
    "primary_uk_sic_2007_code": "string", #                 String to retain leading zeros
    "primary_uk_sic_2007_description": "string",
    "full_overview": "string",
    "history": "string",
    "primary_business_line": "string",
    "secondary_business_line": "string",
    "main_activity": "string",
    "secondary_activity": "string",
    "main_products_and_services": "string",
    "size_estimate": "string",
    
    # Strategy & International Exposure
    "strategy_organization_and_policy": "string",
    "strategic_alliances": "string",
    "membership_of_a_network": "string",
    "main_brand_names": "string",
    "main_domestic_country": "string",
    "main_foreign_countries_or_regions": "string",
    "main_production_sites": "string",
    "main_distribution_sites": "string",
    "main_sales_representation_sites": "string",
    "main_customers": "string",
    
    # Temporal metadata
    "latest_accounts_date": "date",
    "no_of_available_years": "int32",

    # Flags to indicate whether data was available
    "data_av_ptaddress": "boolean",
    "data_av_ptaddress_latlong": "boolean"
})
df_schema_names = set(fame_schema.names)
print(f"Schema column names: {df_schema_names}")

assert len(fame_schema.names) == 51, f"Expected 51 columns in fame_schema, but found {len(fame_schema.names)}"

# 4. Execute the table creation using the Ibis schema
try:
    
    # 2. Connect to DuckDB using Ibis
    con = ibis.duckdb.connect(str(db_path))
    print(f"Initializing DuckDB via Ibis at: {db_path}")
    # overwrite=True prevents errors if the script is run multiple times during setup
    con.create_table("fame_id_data", schema=fame_schema, overwrite=True)
    print("✅ Successfully created Ibis schema for 'fame_id_data'.")
    
    # Optional: Print the table structure to verify
    print("\nTable Schema Verification:")
    print(con.table("fame_id_data").schema())
    
except Exception as e:
    print(f"❌ Error creating table: {e}")

Path: c:\Users\lazym\Documents\Code\dissertation\.venv\Lib\site-packages\ibis\__init__.py
Version: 12.0.0
Schema column names: {'primary_uk_sic_2007_code', 'main_sales_representation_sites', 'ro_address_line_5', 'strategic_alliances', 'main_customers', 'primary_trading_address_longitude', 'main_distribution_sites', 'ro_longitude', 'ro_full_postcode', 'branch_name', 'history', 'secondary_business_line', 'ro_county', 'membership_of_a_network', 'latest_accounts_date', 'ro_phone_registered_on_ctps', 'no_of_available_years', 'main_brand_names', 'main_foreign_countries_or_regions', 'primary_uk_sic_2007_description', 'ro_postcode', 'ticker_symbol', 'data_av_ptaddress_latlong', 'ro_country', 'ro_city', 'ro_phone', 'ro_phone_registered_on_tps', 'registered_number', 'company_name', 'primary_trading_address', 'ro_address', 'ro_address_line_2', 'primary_trading_address_no_of_employees', 'ro_nuts_region', 'secondary_activity', 'data_av_ptaddress', 'full_overview', 'primary_business_line', 'main_dom

# 3. Process each excel file in the raw file dictionary

In [55]:
# Create a dict fuzzy matching the scheme to the following column names in the raw data
# Use an array etc for list of possibilities that the column names could be
# Later we will assume that the columns preserve order but just check that the column names are matching
# And throw up an error if they are not matching.
# Company name	Registered number	Ticker symbol	R/O Address	R/O Address, line 1	R/O Address, line 2	R/O Address, line 3	R/O Address, line 4	R/O Address, line 5	R/O City	R/O County	R/O Postcode	R/O Full Postcode	R/O Country	R/O Latitude	R/O Longitude	R/O NUTS region	R/O Postal region	R/O Phone	R/O Phone registered on TPS	R/O Phone registered on CTPS	Primary trading address	Primary trading address Latitude	Primary trading address Longitude	Primary trading address No of employees	Branch name	Trade description	Primary UK SIC (2007) code	Primary UK SIC (2007) description	Full overview	History	Primary business line	Secondary business line	Main activity	Secondary activity	Main products and services	Size estimate	Strategy, organization and policy	Strategic alliances	Membership of a network	Main brand names	Main domestic country	Main foreign countries or regions	Main production sites	Main distribution sites	Main sales representation sites	Main customers	Latest accounts date	No of available years
fuzzy_col_mapping = {
    "company_name": ["Company name"],
    "registered_number": ["Registered number"],
    "ticker_symbol": ["Ticker symbol"],
    "ro_address": ["R/O Address"],
    "ro_address_line_1": ["R/O Address, line 1"],
    "ro_address_line_2": ["R/O Address, line 2"],
    "ro_address_line_3": ["R/O Address, line 3"],
    "ro_address_line_4": ["R/O Address, line 4"],
    "ro_address_line_5": ["R/O Address, line 5"],
    "ro_city": ["R/O City"],
    "ro_county": ["R/O County"],
    "ro_postcode": ["R/O Postcode"],
    "ro_full_postcode": ["R/O Full Postcode"],
    "ro_country": ["R/O Country"],
    "ro_latitude": ["R/O Latitude"],
    "ro_longitude": ["R/O Longitude"],
    "ro_nuts_region": ["R/O NUTS region"],
    "ro_postal_region": ["R/O Postal region"],
    "ro_phone": ["R/O Phone"],
    "ro_phone_registered_on_tps": ["R/O Phone registered on TPS"],
    "ro_phone_registered_on_ctps": ["R/O Phone registered on CTPS"],
    "primary_trading_address": ["Primary trading address"],
    "primary_trading_address_latitude": ["Primary trading address Latitude"],
    "primary_trading_address_longitude": ["Primary trading address Longitude"],
    "primary_trading_address_no_of_employees": ["Primary trading address No of employees"],
    "branch_name": ["Branch name"],
    "trade_description": ["Trade description"],
    "primary_uk_sic_2007_code": ["Primary UK SIC (2007) code"],
    "primary_uk_sic_2007_description": ["Primary UK SIC (2007) description"],
    "full_overview": ["Full overview"],
    "history": ["History"],
    "primary_business_line": ["Primary business line"],
    "secondary_business_line": ["Secondary business line"],
    "main_activity": ["Main activity"],
    "secondary_activity": ["Secondary activity"],
    "main_products_and_services": ["Main products and services"],
    "size_estimate": ["Size estimate"],
    "strategy_organization_and_policy": ["Strategy, organization and policy"],
    "strategic_alliances": ["Strategic alliances"],
    "membership_of_a_network": ["Membership of a network"],
    "main_brand_names": ["Main brand names"],
    "main_domestic_country": ["Main domestic country"],
    "main_foreign_countries_or_regions": ["Main foreign countries or regions"],
    "main_production_sites": ["Main production sites"],
    "main_distribution_sites": ["Main distribution sites"],
    "main_sales_representation_sites": ["Main sales representation sites"],
    "main_customers": ["Main customers"],
    "latest_accounts_date": ["Latest accounts date"],
    "no_of_available_years": ["No of available years"]
}

# Column number -> column name mapping # ex: 0 -> "Company name"
fuzzy_col_by_index = {i: fuzzy_col_mapping[key][0] for i, key in enumerate(fuzzy_col_mapping.keys())}

# Raw column name -> FAME schema name mapping, based on every possibility of the raw column names
fuzzy_col_to_schema_mapping = {raw_name: schema_name for schema_name, raw_names in fuzzy_col_mapping.items() for raw_name in raw_names}

def check_column_name(col_index: int, actual_col_name: str, test_mode: bool = False) -> bool:
    expected_col_name = fuzzy_col_by_index.get(col_index)
    if expected_col_name is None:
        raise ValueError(f"Column index {col_index} is out of range.")
    
    if actual_col_name != expected_col_name:
        if not test_mode:
            raise ValueError(f"❌ Column mismatch at index {col_index}: Expected '{expected_col_name}', but got '{actual_col_name}'")
        return False
    
    return True

def check_column_names_in_dataframe(df: pd.DataFrame, test_mode: bool = False) -> bool:
    for i, actual_col_name in enumerate(df.columns):
        if not check_column_name(i, actual_col_name, test_mode=test_mode):
            return False
    return True

def check_df_matches_schema(df: pd.DataFrame, test_mode: bool = False) -> bool:

    # For each column in the DataFrame, check if the column name exists in the fame.schema
    # And tick off every column in fame.schema that is found
    # if any are missing from the given DataFrame, then throw an error
    print(f"Schema column names: {df_schema_names}")
    df_column_names = set(df.columns)
    missing_columns = df_schema_names - df_column_names
    extra_columns = df_column_names - df_schema_names

    # Check for missing or extra columns
    if missing_columns:
        if not test_mode:
            raise ValueError(f"❌ Missing columns in DataFrame: {missing_columns}")
        return False

    if extra_columns:
        if not test_mode:
            raise ValueError(f"❌ Extra columns in DataFrame: {extra_columns}")
        return False

    # Check if the number of columns matches
    if len(df.columns) != len(fuzzy_col_by_index):
        if not test_mode:
            raise ValueError(f"❌ Column count mismatch: Expected {len(fuzzy_col_by_index)} columns, but got {len(df.columns)}")
        return False
    
    return True

    # The above function when I run it with real data, throws up the following error:
    # ❌ Error processing file 88/a1_ID/Export 21_02_2025 16_52.xlsx
    # Error: ❌ Extra columns in DataFrame: {'data_av_ptaddress', 'data_av_ptaddress_latlong'}
    # But these columns are in the fame.schema, so why is it throwing up this error?
    # I think it is because the fame.schema is not being used to check the DataFrame, but rather the fuzzy_col_by_index dict.
    # So I need to change the function to use the fame.schema instead of the fuzzy_col_by_index dict.
    # But I can literally see that df_scheme_names = set(fame_schema.names). So that can't be right.
    # Instead the answer must be: 


# Set flags in the DataFrame to indicate whether data was available for certain columns.
def set_data_av_flags(df: pd.DataFrame) -> pd.DataFrame:
    df['data_av_ptaddress'] = df['primary_trading_address'].notna()
    df['data_av_ptaddress_latlong'] = df['primary_trading_address_latitude'].notna() & df['primary_trading_address_longitude'].notna()
    return df

# Generate tests for the above
def test_check_column_name():
    # Test with correct column names
    for i, expected_col_name in fuzzy_col_by_index.items():
        assert check_column_name(i, expected_col_name, test_mode=True) == True, f"Test failed for index {i} with correct name '{expected_col_name}'"
    
    # Test with incorrect column names
    for i, expected_col_name in fuzzy_col_by_index.items():
        incorrect_name = expected_col_name + "_wrong"
        assert check_column_name(i, incorrect_name, test_mode=True) == False, f"Test failed for index {i} with incorrect name '{incorrect_name}'"
    
    # Test with out of range index
    try:
        check_column_name(len(fuzzy_col_by_index), "Some Name")
    except ValueError as e:
        assert str(e) == f"Column index {len(fuzzy_col_by_index)} is out of range.", "Test failed for out of range index"

    print("✅ All tests passed for check_column_name function.")

def test_check_column_names_in_dataframe():
    # Create a DataFrame with correct column names
    correct_df = pd.DataFrame(columns=[fuzzy_col_by_index[i] for i in range(len(fuzzy_col_by_index))])
    assert check_column_names_in_dataframe(correct_df) == True, "Test failed for DataFrame with correct column names"
    
    # Create a DataFrame with incorrect column names
    incorrect_df = pd.DataFrame(columns=[fuzzy_col_by_index[i] + "_wrong" for i in range(len(fuzzy_col_by_index))])
    assert check_column_names_in_dataframe(incorrect_df, test_mode=True) == False, "Test failed for DataFrame with incorrect column names"

    print("✅ All tests passed for check_column_names_in_dataframe function.")

# Tests for fuzzy_col_to_schema_mapping
def test_fuzzy_col_to_schema_mapping():
    # Test that all raw names map to the correct schema name
    for schema_name, raw_names in fuzzy_col_mapping.items():
        for raw_name in raw_names:
            assert fuzzy_col_to_schema_mapping[raw_name] == schema_name, f"Test failed for raw name '{raw_name}' mapping to schema name '{schema_name}'"
    
    # Test a few hardcoded examples
    assert fuzzy_col_to_schema_mapping["Company name"] == "company_name", "Test failed for 'Company name'"
    assert fuzzy_col_to_schema_mapping["R/O Address, line 1"] == "ro_address_line_1", "Test failed for 'R/O Address, line 1'"
    assert fuzzy_col_to_schema_mapping["Primary trading address Latitude"] == "primary_trading_address_latitude", "Test failed for 'Primary trading address Latitude'"
    assert fuzzy_col_to_schema_mapping["Latest accounts date"] == "latest_accounts_date", "Test failed for 'Latest accounts date'"
    
    print("✅ All tests passed for fuzzy_col_to_schema_mapping.")

# Tests for set_data_av_flags
def test_set_data_av_flags():
    # Create a DataFrame with some missing values
    test_data = {
        "primary_trading_address": ["Address 1", None, "Address 3"],
        "primary_trading_address_latitude": ["Lat 1", None, "Lat 3"],
        "primary_trading_address_longitude": ["Long 1", None, "Long 3"]
    }
    df = pd.DataFrame(test_data)
    
    # Apply the function
    df_with_flags = set_data_av_flags(df)
    
    # Check the flags
    expected_data_av_ptaddress = [True, False, True]
    expected_data_av_ptaddress_latlong = [True, False, True]
    
    assert df_with_flags['data_av_ptaddress'].tolist() == expected_data_av_ptaddress, "Test failed for 'data_av_ptaddress' flag"
    assert df_with_flags['data_av_ptaddress_latlong'].tolist() == expected_data_av_ptaddress_latlong, "Test failed for 'data_av_ptaddress_latlong' flag"
    
    print("✅ All tests passed for set_data_av_flags.")

# Test for check_df_matches_schema based on the ibis.schema object
# don't use the fuzzy_col_by_index object because it doesn't contain all the columns of the schema
def test_check_df_matches_schema():

    # Create a DataFrame with correct column names
    correct_df = pd.DataFrame(columns=fame_schema.names)
    assert check_df_matches_schema(correct_df) == True, "Test failed for DataFrame with correct schema"
    
    # Create a DataFrame with missing columns
    incorrect_df_missing = pd.DataFrame(columns=fame_schema.names[:-1])  # Remove last column
    assert check_df_matches_schema(incorrect_df_missing, test_mode=True) == False, "Test failed for DataFrame with missing columns"
    
    # Create a DataFrame with extra columns
    incorrect_df_extra = pd.DataFrame(columns=list(fame_schema.names) + ["extra_column"])
    assert check_df_matches_schema(incorrect_df_extra, test_mode=True) == False, "Test failed for DataFrame with extra columns"
    
    # Create a DataFrame with incorrect column count (fewer columns)
    incorrect_df_count = pd.DataFrame(columns=[col for col in fame_schema.names[:-1]])  # Remove last column
    assert check_df_matches_schema(incorrect_df_count, test_mode=True) == False, "Test failed for DataFrame with incorrect column count"

    # Hardcode a test for all the column names in the scheme in a dataframe and check that the function returns True
    hard_names = [
                    "company_name", "registered_number", "ticker_symbol", "ro_address", "ro_address_line_1", "ro_address_line_2", "ro_address_line_3",
                    "ro_address_line_4", "ro_address_line_5", "ro_city", "ro_county", "ro_postcode", "ro_full_postcode", "ro_country", "ro_latitude", "ro_longitude",
                    "ro_nuts_region", "ro_postal_region", "ro_phone", "ro_phone_registered_on_tps", "ro_phone_registered_on_ctps", "primary_trading_address", "primary_trading_address_latitude",
                    "primary_trading_address_longitude", "primary_trading_address_no_of_employees", "branch_name", "trade_description", "primary_uk_sic_2007_code",
                    "primary_uk_sic_2007_description", "full_overview", "history", "primary_business_line", "secondary_business_line", "main_activity", "secondary_activity",
                    "main_products_and_services", "size_estimate", "strategy_organization_and_policy", "strategic_alliances", "membership_of_a_network", "main_brand_names",
                    "main_domestic_country", "main_foreign_countries_or_regions", "main_production_sites", "main_distribution_sites", "main_sales_representation_sites", "main_customers"
                    , "latest_accounts_date", "no_of_available_years", "data_av_ptaddress", "data_av_ptaddress_latlong"
                ]
    hardcoded_df = pd.DataFrame(columns=hard_names)
    assert check_df_matches_schema(hardcoded_df) == True, "Test failed for hardcoded DataFrame with correct schema"

    print("✅ All tests passed for check_df_matches_schema.")

test_check_column_name()
test_check_column_names_in_dataframe()
test_fuzzy_col_to_schema_mapping()
test_set_data_av_flags()
test_check_df_matches_schema()

✅ All tests passed for check_column_name function.
✅ All tests passed for check_column_names_in_dataframe function.
✅ All tests passed for fuzzy_col_to_schema_mapping.
✅ All tests passed for set_data_av_flags.
Schema column names: {'primary_uk_sic_2007_code', 'main_sales_representation_sites', 'ro_address_line_5', 'strategic_alliances', 'main_customers', 'primary_trading_address_longitude', 'main_distribution_sites', 'ro_longitude', 'ro_full_postcode', 'branch_name', 'history', 'secondary_business_line', 'ro_county', 'membership_of_a_network', 'latest_accounts_date', 'ro_phone_registered_on_ctps', 'no_of_available_years', 'main_brand_names', 'main_foreign_countries_or_regions', 'primary_uk_sic_2007_description', 'ro_postcode', 'ticker_symbol', 'data_av_ptaddress_latlong', 'ro_country', 'ro_city', 'ro_phone', 'ro_phone_registered_on_tps', 'registered_number', 'company_name', 'primary_trading_address', 'ro_address', 'ro_address_line_2', 'primary_trading_address_no_of_employees', 'ro_nuts

ValueError: ❌ Column count mismatch: Expected 49 columns, but got 51

In [1]:
# 3. Process each excel file in the raw file dictionary
# Extract the data from each Excel file from tab 'Results'
# Where headers are in row 1 and rows are below that
# And the first column 'Company name' is column B
# Enter the data into the above fame_schema and console.log it

# Traverse the raw_file_dict.json file to get each Excel filepath
# declare raw_file_dict as a RawFileDict type
raw_file_dict: RawFileDict | None = None

with open(dirs.output_dir / "raw_file_dict.json", "r") as f:
    raw_file_dict = json.load(f)

if raw_file_dict is None:
    raise ValueError("❌ Error: raw_file_dict.json is empty or not found.")

process_count = 0

# We want one big dataframe, which we will merge all the data into for now
df = pd.DataFrame(columns=list(fame_schema.names))
for ind, obj in raw_file_dict.items():
    for property, arr in obj.items():

        if process_count >= 3:
            break

        if property != "a1_ID":
            continue

        [file_name, file_path] = arr

        try:

            # Read the Excel file, assuming the relevant data is in the 'Results' sheet
            # Drop column A
            # For each raw colum name, map it to the fame_schema name using fuzzy_col_to_schema_mapping
            # Set the data availability flags
            file_df = pd.read_excel(file_path, sheet_name='Results', header=0)
            file_df.drop(file_df.columns[0], axis=1, inplace=True)
            file_df.rename(columns=fuzzy_col_to_schema_mapping, inplace=True)
            flagged_df = set_data_av_flags(file_df)
            check_df_matches_schema(flagged_df)

            # Merge the current file_df into the main df
            df = pd.concat([df, file_df], ignore_index=True)

            # # Insert data into DuckDB table
            # con.insert("fame_id_data", file_df)
            print(f"✅ Successfully processed and inserted data from: {ind}/{property}/{file_name}")
            
        except Exception as e:
            print(f"❌ Error processing file {ind}/{property}/{file_name}")
            print("Error:", e)
            print("\n")

        process_count += 1

print(df)

NameError: name 'dirs' is not defined